# 06 — Training-Set Augmentation Benchmark

This notebook evaluates whether training-set augmentation changes discrimination on the fixed athlete-disjoint landmark task.

## Scientific question

The augmentation study does **not** create new independent athletes, injury events, or biological information. Synthetic observations are used only inside outer-training folds. Every reported performance estimate is computed exclusively on real held-out athlete-sessions.

The prespecified augmentation arms are:

- **NONE:** no augmentation;
- **SMOTE:** training-only interpolation after training-only median imputation and standardization;
- **CTGAN:** synthetic positive observations generated from real positive training observations only.

The prespecified post-augmentation positive-to-negative ratios are `0.025`, `0.05`, and `0.10`, with stochastic seeds `11, 22, 33, 44, 55`.

Two downstream learners are retained:

- Logistic Regression;
- Random Forest.

This deliberately avoids an unconstrained model-by-augmentation search.

## Interpretation

The target remains the session-level injury-associated indicator. Exact within-session injury-onset timestamps are unavailable. Augmentation results therefore do not support minute-specific injury-risk, onset-localization, causal, or synthetic-data-fidelity claims.

The inferential ceiling remains the five athletes contributing positive sessions.


In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import platform
import random
import warnings

import imblearn
import numpy as np
import pandas as pd
import sdv
import sklearn
import torch

from imblearn.over_sampling import SMOTE
from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 200)

# Preserve the historical Logistic Regression specification exactly while
# suppressing the scikit-learn 1.9 deprecation warning for `penalty="l2"`.
warnings.filterwarnings(
    "ignore",
    message="'penalty' was deprecated",
    category=FutureWarning,
)

print("Python:", platform.python_version())
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("imbalanced-learn:", imblearn.__version__)
print("SDV:", sdv.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## 1. Repository paths and canonical upstream artifacts

Notebook 03 supplies the modelling resource and the ordered 63-feature CUM+DYN representation. Notebook 04 supplies the frozen outer-fold athlete assignments.

The full augmentation grid is checkpointed under `results/augmentation/`. If the checkpoint is absent, the notebook regenerates the full grid. A publication run must never silently relabel an unfingerprinted legacy checkpoint.


In [ ]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    if start.name.lower() == "notebooks":
        return start.parent
    return start


PROJECT_ROOT = find_project_root(Path.cwd())

MODELLING_DIR = PROJECT_ROOT / "results" / "modelling_data"
BASELINE_DIR = PROJECT_ROOT / "results" / "baseline"
OUTPUT_DIR = PROJECT_ROOT / "results" / "augmentation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FILE = MODELLING_DIR / "model_df_2020.csv"
FEATURE_FILE = MODELLING_DIR / "primary_cum_dyn_features_2020.csv"
FOLD_FILE = BASELINE_DIR / "outer_fold_athlete_assignments.csv"

OOF_FILE = OUTPUT_DIR / "augmentation_full_oof.csv"
CHECKPOINT_META_FILE = OUTPUT_DIR / "augmentation_checkpoint_metadata.json"

for required_file in [MODEL_FILE, FEATURE_FILE, FOLD_FILE]:
    if not required_file.exists():
        raise FileNotFoundError(
            "Required upstream artifact is missing. "
            "Run Notebooks 03 and 04 before Notebook 06. "
            f"Missing: {required_file}"
        )

print("Project root:", PROJECT_ROOT)
print("Augmentation output directory:", OUTPUT_DIR)
print("OOF checkpoint:", OOF_FILE)


## 2. Load and validate cohort, primary features, and frozen folds


In [ ]:
df = pd.read_csv(MODEL_FILE, low_memory=False)

feature_table = (
    pd.read_csv(FEATURE_FILE)
    .sort_values("feature_order")
    .reset_index(drop=True)
)

primary_features = feature_table["feature_name"].tolist()

fold_assignment = (
    pd.read_csv(FOLD_FILE)
    .sort_values(
        ["test_fold", "positive_athlete", "player_name"],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)

assert len(primary_features) == 63
assert len(set(primary_features)) == 63
assert set(primary_features).issubset(df.columns)

df["team"] = (
    df["player_name"]
    .astype(str)
    .str.split("-", n=1)
    .str[0]
)

session_table = (
    df[["player_name", "session_id", "injury", "team"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

team_a_sessions = (
    session_table.loc[
        session_table["team"] == "TeamA"
    ]
    .copy()
)

assert len(session_table) == 3_743
assert session_table["player_name"].nunique() == 48
assert int(session_table["injury"].sum()) == 22

assert len(team_a_sessions) == 2_259
assert team_a_sessions["player_name"].nunique() == 27
assert int(team_a_sessions["injury"].sum()) == 22

assert len(fold_assignment) == 27
assert fold_assignment["player_name"].nunique() == 27
assert fold_assignment["test_fold"].nunique() == 5
assert (
    fold_assignment
    .groupby("test_fold")["positive_athlete"]
    .sum()
    .eq(1)
    .all()
)

team_a_athletes = sorted(
    team_a_sessions["player_name"].unique()
)

outer_folds = []

for fold_id in sorted(fold_assignment["test_fold"].unique()):
    test_athletes = (
        fold_assignment.loc[
            fold_assignment["test_fold"] == fold_id,
            "player_name",
        ]
        .sort_values()
        .tolist()
    )

    train_athletes = [
        athlete
        for athlete in team_a_athletes
        if athlete not in test_athletes
    ]

    assert set(train_athletes).isdisjoint(test_athletes)

    outer_folds.append(
        {
            "fold": int(fold_id),
            "train_athletes": train_athletes,
            "test_athletes": test_athletes,
        }
    )

feature_json = json.dumps(primary_features, ensure_ascii=False)
PRIMARY_FEATURE_FINGERPRINT = hashlib.sha256(
    feature_json.encode("utf-8")
).hexdigest()[:16]

fold_json = fold_assignment.to_json(
    orient="records",
    date_format="iso",
)
FOLD_ASSIGNMENT_FINGERPRINT = hashlib.sha256(
    fold_json.encode("utf-8")
).hexdigest()[:16]

print("Primary feature fingerprint:", PRIMARY_FEATURE_FINGERPRINT)
print("Fold assignment fingerprint:", FOLD_ASSIGNMENT_FINGERPRINT)
print("Canonical cohort/features/folds validated.")


## 3. Freeze the augmentation experiment configuration

The row-level experiment fingerprint intentionally preserves the configuration used for the completed benchmark. Feature identity is included directly in the configuration. Fold identity is additionally recorded through a separate fold-assignment fingerprint.


In [ ]:
LANDMARKS = [10, 20, 30, 40, 50, 60]
AUGMENTATION_METHODS = ["smote", "ctgan"]
AUGMENTATION_RATIOS = [0.025, 0.05, 0.10]
AUGMENTATION_SEEDS = [11, 22, 33, 44, 55]
AUGMENTATION_MODELS = ["logistic", "random_forest"]

EXPERIMENT_CONFIG = {
    "landmarks": LANDMARKS,
    "augmentation_methods": AUGMENTATION_METHODS,
    "augmentation_ratios": AUGMENTATION_RATIOS,
    "augmentation_seeds": AUGMENTATION_SEEDS,
    "augmentation_models": AUGMENTATION_MODELS,
    "n_outer_folds": 5,
    "primary_feature_count": len(primary_features),
    "primary_features": primary_features,
    "ctgan_epochs": 100,
    "smote_preprocessing": (
        "training_median_imputation_then_standard_scaling"
    ),
    "ctgan_training_data": "real_positive_training_rows_only",
    "logistic": {
        "C": 1.0,
        "class_weight": "balanced",
        "solver": "liblinear",
        "max_iter": 2000,
        "random_state": 42,
    },
    "random_forest": {
        "n_estimators": 500,
        "max_depth": None,
        "min_samples_leaf": 2,
        "class_weight": "balanced",
        "random_state": 42,
    },
    "versions": {
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "sklearn": sklearn.__version__,
        "imbalanced_learn": imblearn.__version__,
        "sdv": sdv.__version__,
    },
}

config_json = json.dumps(
    EXPERIMENT_CONFIG,
    sort_keys=True,
    default=str,
)

EXPERIMENT_FINGERPRINT = hashlib.sha256(
    config_json.encode("utf-8")
).hexdigest()[:16]

checkpoint_metadata = {
    "experiment_fingerprint": EXPERIMENT_FINGERPRINT,
    "primary_feature_fingerprint": PRIMARY_FEATURE_FINGERPRINT,
    "fold_assignment_fingerprint": FOLD_ASSIGNMENT_FINGERPRINT,
    "experiment_config": EXPERIMENT_CONFIG,
}

if CHECKPOINT_META_FILE.exists():
    with open(CHECKPOINT_META_FILE, "r", encoding="utf-8") as file:
        existing_metadata = json.load(file)

    if (
        existing_metadata.get("experiment_fingerprint")
        != EXPERIMENT_FINGERPRINT
    ):
        raise RuntimeError(
            "Checkpoint metadata fingerprint does not match the "
            "current experiment configuration."
        )
else:
    with open(CHECKPOINT_META_FILE, "w", encoding="utf-8") as file:
        json.dump(
            checkpoint_metadata,
            file,
            indent=2,
            sort_keys=True,
            default=str,
        )

print("Experiment fingerprint:", EXPERIMENT_FINGERPRINT)
print("Experiment configuration frozen.")


## 4. Split-first fold helper

The athlete split occurs before any imputation, scaling, SMOTE, CTGAN fitting, or downstream model fitting.


In [ ]:
def get_real_fold_data(
    landmark: int,
    fold_index: int,
):
    fold = outer_folds[fold_index]

    landmark_df = (
        df.loc[
            (df["minute_idx"] == landmark)
            & (df["player_name"].isin(team_a_athletes))
        ]
        .copy()
        .reset_index(drop=True)
    )

    train_df = landmark_df.loc[
        landmark_df["player_name"].isin(
            fold["train_athletes"]
        )
    ].copy()

    test_df = landmark_df.loc[
        landmark_df["player_name"].isin(
            fold["test_athletes"]
        )
    ].copy()

    assert set(train_df["player_name"]).isdisjoint(
        set(test_df["player_name"])
    )

    X_train = train_df[primary_features].copy()
    y_train = train_df["injury"].astype(int).copy()

    X_test = test_df[primary_features].copy()
    y_test = test_df["injury"].astype(int).copy()

    assert y_train.nunique() == 2
    assert y_test.nunique() == 2

    return {
        "train_df": train_df,
        "test_df": test_df,
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test,
    }


check = get_real_fold_data(
    landmark=30,
    fold_index=0,
)

assert check["X_train"].shape == (1828, 63)
assert int(check["y_train"].sum()) == 18
assert check["X_test"].shape == (429, 63)
assert int(check["y_test"].sum()) == 4

print("30-min Fold-1 train:", check["X_train"].shape)
print("30-min Fold-1 test:", check["X_test"].shape)
print("Split-first athlete isolation validated.")


## 5. Canonical training-only augmentation functions

SMOTE is fit only after training-only median imputation and standardization. Only newly generated SMOTE observations are inverse-transformed and appended; all original real training rows are preserved exactly.

CTGAN is fit exclusively on real positive observations from the outer-training fold.


In [ ]:
def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


def augment_training_data(
    X_train,
    y_train,
    method="none",
    sampling_strategy=0.05,
    random_state=42,
    ctgan_epochs=100,
):
    X_train = X_train.copy()
    y_train = y_train.copy()

    if method == "none":
        return X_train, y_train

    if method == "smote":
        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()

        X_imp = imputer.fit_transform(X_train)
        X_scaled = scaler.fit_transform(X_imp)

        smote = SMOTE(
            sampling_strategy=sampling_strategy,
            random_state=random_state,
        )

        X_res_scaled, y_res = smote.fit_resample(
            X_scaled,
            y_train,
        )

        n_original = len(X_train)
        X_syn_scaled = X_res_scaled[n_original:]

        if len(X_syn_scaled) == 0:
            return X_train, y_train

        X_syn_raw = scaler.inverse_transform(X_syn_scaled)

        X_syn = pd.DataFrame(
            X_syn_raw,
            columns=X_train.columns,
        )

        y_syn = pd.Series(
            np.ones(len(X_syn), dtype=int),
            name=y_train.name,
        )

        X_aug = pd.concat(
            [
                X_train.reset_index(drop=True),
                X_syn,
            ],
            ignore_index=True,
        )

        y_aug = pd.concat(
            [
                y_train.reset_index(drop=True),
                y_syn,
            ],
            ignore_index=True,
        )

        return X_aug, y_aug

    if method == "ctgan":
        set_all_seeds(random_state)

        positive_real = (
            X_train.loc[y_train.to_numpy() == 1]
            .copy()
            .reset_index(drop=True)
        )

        n_negative = int((y_train == 0).sum())

        target_positive = int(
            round(sampling_strategy * n_negative)
        )

        n_to_generate = max(
            0,
            target_positive - len(positive_real),
        )

        if n_to_generate == 0:
            return X_train, y_train

        metadata = SingleTableMetadata()
        metadata.detect_from_dataframe(positive_real)

        synthesizer = CTGANSynthesizer(
            metadata,
            epochs=ctgan_epochs,
            verbose=False,
        )

        synthesizer.fit(positive_real)

        X_syn = synthesizer.sample(
            num_rows=n_to_generate
        )

        y_syn = pd.Series(
            np.ones(n_to_generate, dtype=int),
            name=y_train.name,
        )

        X_aug = pd.concat(
            [
                X_train.reset_index(drop=True),
                X_syn.reset_index(drop=True),
            ],
            ignore_index=True,
        )

        y_aug = pd.concat(
            [
                y_train.reset_index(drop=True),
                y_syn,
            ],
            ignore_index=True,
        )

        return X_aug, y_aug

    raise ValueError(
        "method must be one of: none, smote, ctgan"
    )


## 6. Augmentation implementation and CTGAN quality audit

The audit is intentionally diagnostic. It checks training-only behavior, stochastic reproducibility, nearest-neighbor memorization signals, univariate drift, and dependence drift. Passing the audit does not establish synthetic-data fidelity.


In [ ]:
# SMOTE implementation check
X_smote, y_smote = augment_training_data(
    check["X_train"],
    check["y_train"],
    method="smote",
    sampling_strategy=0.05,
    random_state=11,
)

original_preserved = (
    X_smote.iloc[: len(check["X_train"])]
    .reset_index(drop=True)
    .equals(
        check["X_train"].reset_index(drop=True)
    )
)

assert original_preserved

# CTGAN same-seed reproducibility and cross-seed diversity
def fit_sample_ctgan_for_audit(seed, n_rows=200):
    set_all_seeds(seed)

    positive_real = (
        check["X_train"].loc[
            check["y_train"].to_numpy() == 1
        ]
        .copy()
        .reset_index(drop=True)
    )

    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(positive_real)

    synthesizer = CTGANSynthesizer(
        metadata,
        epochs=100,
        verbose=False,
    )

    synthesizer.fit(positive_real)
    return synthesizer.sample(num_rows=n_rows)


syn_11_a = fit_sample_ctgan_for_audit(11, n_rows=200)
syn_11_b = fit_sample_ctgan_for_audit(11, n_rows=200)
syn_22 = fit_sample_ctgan_for_audit(22, n_rows=200)

assert np.allclose(
    syn_11_a.to_numpy(dtype=float),
    syn_11_b.to_numpy(dtype=float),
    equal_nan=True,
)

assert not np.allclose(
    syn_11_a.to_numpy(dtype=float),
    syn_22.to_numpy(dtype=float),
    equal_nan=True,
)

real_positive = (
    check["X_train"].loc[
        check["y_train"].to_numpy() == 1
    ]
    .copy()
    .reset_index(drop=True)
)

scaler = StandardScaler()
real_scaled = scaler.fit_transform(real_positive)
syn_scaled = scaler.transform(syn_11_a)

nn_real = NearestNeighbors(n_neighbors=2).fit(real_scaled)
real_distances, _ = nn_real.kneighbors(real_scaled)
real_to_real = real_distances[:, 1]

nn_syn = NearestNeighbors(n_neighbors=1).fit(real_scaled)
syn_distances, _ = nn_syn.kneighbors(syn_scaled)
syn_to_real = syn_distances[:, 0]

fidelity_rows = []

for feature in primary_features:
    real_values = real_positive[feature].astype(float)
    synthetic_values = syn_11_a[feature].astype(float)

    pooled_sd = np.sqrt(
        (
            real_values.var(ddof=1)
            + synthetic_values.var(ddof=1)
        )
        / 2
    )

    smd = (
        np.nan
        if pooled_sd == 0 or np.isnan(pooled_sd)
        else (
            synthetic_values.mean()
            - real_values.mean()
        )
        / pooled_sd
    )

    fidelity_rows.append(
        {
            "feature": feature,
            "abs_smd": abs(smd) if not np.isnan(smd) else np.nan,
        }
    )

fidelity = pd.DataFrame(fidelity_rows)

real_corr = real_positive.corr(numeric_only=True)
syn_corr = syn_11_a.corr(numeric_only=True)

upper_triangle = np.triu(
    np.ones(real_corr.shape, dtype=bool),
    k=1,
)

corr_diff = (
    real_corr.to_numpy()
    - syn_corr.to_numpy()
)

abs_corr_diff = np.abs(
    corr_diff[upper_triangle]
)

ctgan_quality_summary = pd.DataFrame(
    [
        {
            "real_positive_rows": len(real_positive),
            "synthetic_rows": len(syn_11_a),
            "exact_or_near_duplicates_lt_1e-6": int(
                np.sum(syn_to_real < 1e-6)
            ),
            "very_close_lt_0_1": int(
                np.sum(syn_to_real < 0.1)
            ),
            "median_real_to_real_distance": float(
                np.median(real_to_real)
            ),
            "median_synthetic_to_real_distance": float(
                np.median(syn_to_real)
            ),
            "median_distance_ratio": float(
                np.median(syn_to_real)
                / np.median(real_to_real)
            ),
            "median_abs_smd": float(
                fidelity["abs_smd"].median()
            ),
            "features_abs_smd_gt_0_5": int(
                (fidelity["abs_smd"] > 0.5).sum()
            ),
            "features_abs_smd_gt_1_0": int(
                (fidelity["abs_smd"] > 1.0).sum()
            ),
            "median_abs_correlation_difference": float(
                np.median(abs_corr_diff)
            ),
            "mean_abs_correlation_difference": float(
                np.mean(abs_corr_diff)
            ),
            "correlation_pairs_abs_diff_gt_0_25": int(
                np.sum(abs_corr_diff > 0.25)
            ),
            "correlation_pairs_abs_diff_gt_0_50": int(
                np.sum(abs_corr_diff > 0.50)
            ),
        }
    ]
)

display(ctgan_quality_summary)

print(
    "CTGAN audit completed. "
    "Interpret fidelity cautiously because only 18 real positive "
    "training observations are available in this audit fold."
)

del syn_11_b, syn_22
gc.collect()


## 7. Frozen downstream model factory


In [ ]:
def make_model(model_name):
    if model_name == "logistic":
        return Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                (
                    "clf",
                    LogisticRegression(
                        penalty="l2",
                        C=1.0,
                        class_weight="balanced",
                        solver="liblinear",
                        max_iter=2000,
                        random_state=42,
                    ),
                ),
            ]
        )

    if model_name == "random_forest":
        return Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                (
                    "clf",
                    RandomForestClassifier(
                        n_estimators=500,
                        max_depth=None,
                        min_samples_leaf=2,
                        class_weight="balanced",
                        random_state=42,
                        n_jobs=-1,
                    ),
                ),
            ]
        )

    raise ValueError(
        "model_name must be 'logistic' or 'random_forest'"
    )


print("Frozen downstream learners:", AUGMENTATION_MODELS)


## 8. Full-grid experiment runner and checkpoint logic

The full grid contains 900 augmented training units plus 30 deterministic NONE units, for 930 distinct train datasets and 1,860 downstream model fits.

The same augmented training set is reused across the two downstream learners within an experiment unit. Held-out athlete-sessions are never augmented.

If a valid row-fingerprinted checkpoint already exists, completed units are skipped. If no checkpoint exists, the entire grid is generated.


In [ ]:
def run_one_experiment_unit(
    landmark,
    fold_index,
    method,
    ratio,
    seed,
    ctgan_epochs=100,
):
    fold_data = get_real_fold_data(
        landmark=landmark,
        fold_index=fold_index,
    )

    X_train = fold_data["X_train"].copy()
    y_train = fold_data["y_train"].copy()

    X_test = fold_data["X_test"].copy()
    y_test = fold_data["y_test"].copy()

    X_aug, y_aug = augment_training_data(
        X_train,
        y_train,
        method=method,
        sampling_strategy=ratio,
        random_state=seed,
        ctgan_epochs=ctgan_epochs,
    )

    rows = []

    for model_name in AUGMENTATION_MODELS:
        model = make_model(model_name)
        model.fit(X_aug, y_aug)
        probabilities = model.predict_proba(X_test)[:, 1]

        rows.extend(
            {
                "landmark": landmark,
                "fold": fold_index + 1,
                "method": method,
                "ratio": float(ratio),
                "seed": int(seed),
                "model": model_name,
                "player_name": fold_data["test_df"].iloc[i]["player_name"],
                "session_id": fold_data["test_df"].iloc[i]["session_id"],
                "y_true": int(y_test.iloc[i]),
                "y_prob": float(probabilities[i]),
                "n_train_real": len(X_train),
                "n_train_aug": len(X_aug),
                "experiment_fingerprint": EXPERIMENT_FINGERPRINT,
            }
            for i in range(len(X_test))
        )

    return pd.DataFrame(rows)


def normalized_unit_key(
    landmark,
    fold,
    method,
    ratio,
    seed,
):
    if method == "none":
        return (
            int(landmark),
            int(fold),
            "none",
            0.0,
            0,
        )

    return (
        int(landmark),
        int(fold),
        str(method),
        float(ratio),
        int(seed),
    )


if OOF_FILE.exists():
    existing_oof = pd.read_csv(OOF_FILE)

    if "experiment_fingerprint" not in existing_oof.columns:
        raise RuntimeError(
            "Legacy augmentation checkpoint has no row-level "
            "experiment fingerprint. Do not silently migrate it "
            "inside the publication notebook."
        )

    observed_fingerprints = set(
        existing_oof["experiment_fingerprint"]
        .dropna()
        .astype(str)
        .unique()
    )

    if observed_fingerprints != {EXPERIMENT_FINGERPRINT}:
        raise RuntimeError(
            "Existing augmentation checkpoint fingerprint does "
            "not match the current experiment configuration."
        )

    completed_units = {
        normalized_unit_key(
            row.landmark,
            row.fold,
            row.method,
            row.ratio,
            row.seed,
        )
        for row in (
            existing_oof[
                ["landmark", "fold", "method", "ratio", "seed"]
            ]
            .drop_duplicates()
            .itertuples(index=False)
        )
    }

    print("Existing checkpoint rows:", len(existing_oof))
    print("Completed normalized units:", len(completed_units))
else:
    existing_oof = pd.DataFrame()
    completed_units = set()

    print("No checkpoint found. A fresh full-grid run will start.")


def append_unit_to_checkpoint(unit_df):
    global existing_oof, completed_units

    if existing_oof.empty:
        existing_oof = unit_df.copy()
    else:
        existing_oof = pd.concat(
            [existing_oof, unit_df],
            ignore_index=True,
        )

    existing_oof.to_csv(
        OOF_FILE,
        index=False,
    )

    first = unit_df.iloc[0]

    completed_units.add(
        normalized_unit_key(
            first["landmark"],
            first["fold"],
            first["method"],
            first["ratio"],
            first["seed"],
        )
    )


In [ ]:
# Deterministic NONE units
for landmark in LANDMARKS:
    for fold_index in range(5):
        key = normalized_unit_key(
            landmark,
            fold_index + 1,
            "none",
            0.0,
            0,
        )

        if key not in completed_units:
            unit_df = run_one_experiment_unit(
                landmark=landmark,
                fold_index=fold_index,
                method="none",
                ratio=0.0,
                seed=0,
                ctgan_epochs=100,
            )
            append_unit_to_checkpoint(unit_df)

# Stochastic augmented units
for landmark in LANDMARKS:
    for fold_index in range(5):
        for method in AUGMENTATION_METHODS:
            for ratio in AUGMENTATION_RATIOS:
                for seed in AUGMENTATION_SEEDS:
                    key = normalized_unit_key(
                        landmark,
                        fold_index + 1,
                        method,
                        ratio,
                        seed,
                    )

                    if key in completed_units:
                        continue

                    unit_df = run_one_experiment_unit(
                        landmark=landmark,
                        fold_index=fold_index,
                        method=method,
                        ratio=ratio,
                        seed=seed,
                        ctgan_epochs=100,
                    )

                    append_unit_to_checkpoint(unit_df)

                    print(
                        "Completed:",
                        landmark,
                        fold_index + 1,
                        method,
                        ratio,
                        seed,
                    )

print("Checkpointed full-grid execution complete.")


## 9. Full-grid integrity and frozen NONE baseline


In [ ]:
full_oof = pd.read_csv(OOF_FILE)

prediction_key = [
    "landmark",
    "model",
    "method",
    "ratio",
    "seed",
    "player_name",
    "session_id",
]

assert full_oof.duplicated(prediction_key).sum() == 0

normalized_units = {
    normalized_unit_key(
        row.landmark,
        row.fold,
        row.method,
        row.ratio,
        row.seed,
    )
    for row in (
        full_oof[
            ["landmark", "fold", "method", "ratio", "seed"]
        ]
        .drop_duplicates()
        .itertuples(index=False)
    )
}

assert len(normalized_units) == 930

assert (
    full_oof.loc[
        full_oof["method"] != "none",
        ["landmark", "fold", "method", "ratio", "seed"],
    ]
    .drop_duplicates()
    .shape[0]
    == 900
)

baseline_rows = []

for landmark in LANDMARKS:
    for model_name in AUGMENTATION_MODELS:
        base = full_oof.loc[
            (full_oof["landmark"] == landmark)
            & (full_oof["method"] == "none")
            & (full_oof["model"] == model_name)
        ]

        assert not base.duplicated(
            ["player_name", "session_id"]
        ).any()

        baseline_rows.append(
            {
                "landmark": landmark,
                "model": model_name,
                "roc_auc": roc_auc_score(
                    base["y_true"],
                    base["y_prob"],
                ),
                "pr_auc": average_precision_score(
                    base["y_true"],
                    base["y_prob"],
                ),
            }
        )

baseline_results = pd.DataFrame(baseline_rows)

expected_logistic_roc = np.array(
    [0.499349, 0.556940, 0.607200, 0.427717, 0.401846, 0.367348]
)

expected_logistic_pr = np.array(
    [0.010696, 0.014999, 0.012902, 0.008785, 0.007913, 0.008001]
)

expected_rf_roc = np.array(
    [0.504899, 0.612661, 0.626703, 0.631457, 0.533780, 0.631004]
)

expected_rf_pr = np.array(
    [0.009518, 0.012623, 0.013190, 0.013261, 0.010074, 0.014956]
)

for model_name, expected_roc, expected_pr in [
    ("logistic", expected_logistic_roc, expected_logistic_pr),
    ("random_forest", expected_rf_roc, expected_rf_pr),
]:
    tmp = (
        baseline_results.loc[
            baseline_results["model"] == model_name
        ]
        .sort_values("landmark")
    )

    np.testing.assert_allclose(
        tmp["roc_auc"],
        expected_roc,
        atol=5e-6,
        rtol=0.0,
    )

    np.testing.assert_allclose(
        tmp["pr_auc"],
        expected_pr,
        atol=5e-6,
        rtol=0.0,
    )

display(baseline_results.round(6))

print("OOF rows:", len(full_oof))
print("Normalized experiment units:", len(normalized_units))
print("Full-grid integrity checks passed.")


## 10. Seed-level and seed-aggregated augmentation performance


In [ ]:
performance_rows = []

# NONE baseline: deterministic, one row per model and landmark
for landmark in LANDMARKS:
    for model_name in AUGMENTATION_MODELS:
        d = full_oof.loc[
            (full_oof["landmark"] == landmark)
            & (full_oof["method"] == "none")
            & (full_oof["model"] == model_name)
        ]

        performance_rows.append(
            {
                "landmark": landmark,
                "method": "none",
                "ratio": 0.0,
                "seed": 0,
                "model": model_name,
                "roc_auc": roc_auc_score(
                    d["y_true"],
                    d["y_prob"],
                ),
                "pr_auc": average_precision_score(
                    d["y_true"],
                    d["y_prob"],
                ),
                "n": len(d),
                "positives": int(d["y_true"].sum()),
            }
        )

# Augmented arms
for landmark in LANDMARKS:
    for method in AUGMENTATION_METHODS:
        for ratio in AUGMENTATION_RATIOS:
            for seed in AUGMENTATION_SEEDS:
                for model_name in AUGMENTATION_MODELS:
                    d = full_oof.loc[
                        (full_oof["landmark"] == landmark)
                        & (full_oof["method"] == method)
                        & (full_oof["ratio"] == ratio)
                        & (full_oof["seed"] == seed)
                        & (full_oof["model"] == model_name)
                    ]

                    assert len(d) > 0

                    performance_rows.append(
                        {
                            "landmark": landmark,
                            "method": method,
                            "ratio": ratio,
                            "seed": seed,
                            "model": model_name,
                            "roc_auc": roc_auc_score(
                                d["y_true"],
                                d["y_prob"],
                            ),
                            "pr_auc": average_precision_score(
                                d["y_true"],
                                d["y_prob"],
                            ),
                            "n": len(d),
                            "positives": int(d["y_true"].sum()),
                        }
                    )

performance_summary = pd.DataFrame(performance_rows)

aug_seed_summary = (
    performance_summary.loc[
        performance_summary["method"].isin(
            AUGMENTATION_METHODS
        )
    ]
    .groupby(
        ["landmark", "method", "ratio", "model"],
        as_index=False,
    )
    .agg(
        roc_mean=("roc_auc", "mean"),
        roc_sd=("roc_auc", "std"),
        pr_mean=("pr_auc", "mean"),
        pr_sd=("pr_auc", "std"),
    )
)

display(
    aug_seed_summary.loc[
        aug_seed_summary["landmark"].isin([30, 60])
    ]
    .sort_values(
        ["model", "landmark", "method", "ratio"]
    )
    .round(6)
)


## 11. Paired mean effects versus NONE

For every augmentation condition, the point estimate is the mean performance across the five prespecified augmentation seeds minus the deterministic NONE baseline on the exact same real held-out athlete-sessions.


In [ ]:
KEY_COLS = [
    "player_name",
    "session_id",
    "fold",
    "y_true",
]

paired_rows = []

for landmark in LANDMARKS:
    for model_name in AUGMENTATION_MODELS:
        base = (
            full_oof.loc[
                (full_oof["landmark"] == landmark)
                & (full_oof["method"] == "none")
                & (full_oof["model"] == model_name)
            ]
            .sort_values(KEY_COLS)
            .reset_index(drop=True)
        )

        base_roc = roc_auc_score(
            base["y_true"],
            base["y_prob"],
        )

        base_pr = average_precision_score(
            base["y_true"],
            base["y_prob"],
        )

        for method in AUGMENTATION_METHODS:
            for ratio in AUGMENTATION_RATIOS:
                seed_rocs = []
                seed_prs = []

                for seed in AUGMENTATION_SEEDS:
                    aug = (
                        full_oof.loc[
                            (full_oof["landmark"] == landmark)
                            & (full_oof["method"] == method)
                            & (full_oof["ratio"] == ratio)
                            & (full_oof["seed"] == seed)
                            & (full_oof["model"] == model_name)
                        ]
                        .sort_values(KEY_COLS)
                        .reset_index(drop=True)
                    )

                    assert base[KEY_COLS].equals(
                        aug[KEY_COLS]
                    )

                    seed_rocs.append(
                        roc_auc_score(
                            aug["y_true"],
                            aug["y_prob"],
                        )
                    )

                    seed_prs.append(
                        average_precision_score(
                            aug["y_true"],
                            aug["y_prob"],
                        )
                    )

                paired_rows.append(
                    {
                        "landmark": landmark,
                        "model": model_name,
                        "method": method,
                        "ratio": ratio,
                        "baseline_roc": base_roc,
                        "aug_roc_mean": float(np.mean(seed_rocs)),
                        "delta_roc": float(
                            np.mean(seed_rocs) - base_roc
                        ),
                        "delta_roc_seed_sd": float(
                            np.std(seed_rocs, ddof=1)
                        ),
                        "baseline_pr": base_pr,
                        "aug_pr_mean": float(np.mean(seed_prs)),
                        "delta_pr": float(
                            np.mean(seed_prs) - base_pr
                        ),
                        "delta_pr_seed_sd": float(
                            np.std(seed_prs, ddof=1)
                        ),
                    }
                )

paired_effects = pd.DataFrame(paired_rows)

assert len(paired_effects) == 72

display(
    paired_effects.loc[
        paired_effects["landmark"].isin([30, 40, 60])
    ]
    .sort_values(
        ["model", "landmark", "method", "ratio"]
    )
    .round(6)
)


## 12. Fast weighted metric implementation and equivalence test

Athlete-cluster bootstrap multiplicities are represented as observation weights. The vectorized implementation is validated directly against scikit-learn on 50 weighted bootstrap replicates before use.


In [ ]:
def vectorized_weighted_metrics(
    y_true,
    y_prob,
    weight_matrix,
):
    y = np.asarray(y_true, dtype=int)
    scores = np.asarray(y_prob, dtype=float)
    W = np.asarray(weight_matrix, dtype=float)

    assert y.ndim == 1
    assert scores.shape == y.shape
    assert W.shape[1] == len(y)

    order = np.argsort(-scores, kind="mergesort")

    y_sorted = y[order]
    scores_sorted = scores[order]
    W_sorted = W[:, order]

    tp = np.cumsum(
        W_sorted * y_sorted,
        axis=1,
    )

    fp = np.cumsum(
        W_sorted * (1 - y_sorted),
        axis=1,
    )

    group_ends = np.r_[
        np.flatnonzero(
            scores_sorted[:-1] != scores_sorted[1:]
        ),
        len(scores_sorted) - 1,
    ]

    tp = tp[:, group_ends]
    fp = fp[:, group_ends]

    total_pos = tp[:, -1]
    total_neg = fp[:, -1]

    assert np.all(total_pos > 0)
    assert np.all(total_neg > 0)

    tpr = tp / total_pos[:, None]
    fpr = fp / total_neg[:, None]

    tpr0 = np.concatenate(
        [np.zeros((len(W), 1)), tpr],
        axis=1,
    )
    fpr0 = np.concatenate(
        [np.zeros((len(W), 1)), fpr],
        axis=1,
    )

    roc_auc = np.sum(
        (fpr0[:, 1:] - fpr0[:, :-1])
        * (tpr0[:, 1:] + tpr0[:, :-1])
        / 2.0,
        axis=1,
    )

    precision = np.divide(
        tp,
        tp + fp,
        out=np.zeros_like(tp),
        where=((tp + fp) > 0),
    )

    recall = tp / total_pos[:, None]

    previous_recall = np.concatenate(
        [
            np.zeros((len(W), 1)),
            recall[:, :-1],
        ],
        axis=1,
    )

    pr_auc = np.sum(
        (recall - previous_recall)
        * precision,
        axis=1,
    )

    return roc_auc, pr_auc


test_df = (
    full_oof.loc[
        (full_oof["landmark"] == 30)
        & (full_oof["method"] == "none")
        & (full_oof["model"] == "random_forest")
    ]
    .sort_values(KEY_COLS)
    .reset_index(drop=True)
)

y_metric = test_df["y_true"].to_numpy()
p_metric = test_df["y_prob"].to_numpy()

athletes_metric = np.sort(
    test_df["player_name"].unique()
)

athlete_to_idx = {
    athlete: i
    for i, athlete in enumerate(athletes_metric)
}

row_athlete_idx = (
    test_df["player_name"]
    .map(athlete_to_idx)
    .to_numpy()
)

rng = np.random.default_rng(16001)
weight_rows = []

while len(weight_rows) < 50:
    sampled = rng.integers(
        0,
        len(athletes_metric),
        size=len(athletes_metric),
    )

    counts = np.bincount(
        sampled,
        minlength=len(athletes_metric),
    )

    weights = counts[row_athlete_idx]
    active = weights > 0

    if np.unique(y_metric[active]).size == 2:
        weight_rows.append(weights)

W_test = np.asarray(weight_rows, dtype=float)

roc_vec, pr_vec = vectorized_weighted_metrics(
    y_metric,
    p_metric,
    W_test,
)

roc_ref = np.array(
    [
        roc_auc_score(
            y_metric,
            p_metric,
            sample_weight=w,
        )
        for w in W_test
    ]
)

pr_ref = np.array(
    [
        average_precision_score(
            y_metric,
            p_metric,
            sample_weight=w,
        )
        for w in W_test
    ]
)

np.testing.assert_allclose(
    roc_vec,
    roc_ref,
    atol=1e-12,
    rtol=0.0,
)

np.testing.assert_allclose(
    pr_vec,
    pr_ref,
    atol=1e-12,
    rtol=0.0,
)

print(
    "ROC max absolute difference:",
    f"{np.max(np.abs(roc_vec - roc_ref)):.3e}",
)
print(
    "PR max absolute difference:",
    f"{np.max(np.abs(pr_vec - pr_ref)):.3e}",
)
print("Vectorized weighted metrics validated.")


## 13. Shared-draw athlete-cluster bootstrap

Within each landmark, the same 1,000 athlete-bootstrap draws are used across both models, both augmentation methods, all ratios, and all seeds. The bootstrap evaluates only real held-out OOF predictions.


In [ ]:
N_BOOT = 1000
BOOT_SEED = 2026

bootstrap_rows = []

for landmark in LANDMARKS:
    reference_base = (
        full_oof.loc[
            (full_oof["landmark"] == landmark)
            & (full_oof["method"] == "none")
            & (full_oof["model"] == AUGMENTATION_MODELS[0])
        ]
        .sort_values(KEY_COLS)
        .reset_index(drop=True)
    )

    athletes = np.sort(
        reference_base["player_name"].unique()
    )

    athlete_to_idx = {
        athlete: i
        for i, athlete in enumerate(athletes)
    }

    row_athlete_idx = (
        reference_base["player_name"]
        .map(athlete_to_idx)
        .to_numpy()
    )

    reference_y = reference_base["y_true"].to_numpy()

    landmark_rng = np.random.default_rng(
        np.random.SeedSequence(
            [BOOT_SEED, int(landmark)]
        )
    )

    boot_counts = np.zeros(
        (N_BOOT, len(athletes)),
        dtype=np.int16,
    )

    for b in range(N_BOOT):
        sampled_idx = landmark_rng.integers(
            0,
            len(athletes),
            size=len(athletes),
        )

        boot_counts[b] = np.bincount(
            sampled_idx,
            minlength=len(athletes),
        )

    boot_weights = boot_counts[:, row_athlete_idx]

    valid_boot = np.array(
        [
            b
            for b in range(N_BOOT)
            if np.unique(
                reference_y[boot_weights[b] > 0]
            ).size
            == 2
        ],
        dtype=int,
    )

    W = boot_weights[valid_boot]

    print(
        f"Landmark {landmark}: "
        f"{len(valid_boot)}/{N_BOOT} valid bootstrap replicates"
    )

    for model_name in AUGMENTATION_MODELS:
        base = (
            full_oof.loc[
                (full_oof["landmark"] == landmark)
                & (full_oof["method"] == "none")
                & (full_oof["model"] == model_name)
            ]
            .sort_values(KEY_COLS)
            .reset_index(drop=True)
        )

        assert reference_base[KEY_COLS].equals(
            base[KEY_COLS]
        )

        y_true = base["y_true"].to_numpy()
        base_prob = base["y_prob"].to_numpy()

        base_roc_boot, base_pr_boot = (
            vectorized_weighted_metrics(
                y_true,
                base_prob,
                W,
            )
        )

        for method in AUGMENTATION_METHODS:
            for ratio in AUGMENTATION_RATIOS:
                seed_roc_boot = []
                seed_pr_boot = []

                for seed in AUGMENTATION_SEEDS:
                    aug = (
                        full_oof.loc[
                            (full_oof["landmark"] == landmark)
                            & (full_oof["method"] == method)
                            & (full_oof["ratio"] == ratio)
                            & (full_oof["seed"] == seed)
                            & (full_oof["model"] == model_name)
                        ]
                        .sort_values(KEY_COLS)
                        .reset_index(drop=True)
                    )

                    assert base[KEY_COLS].equals(
                        aug[KEY_COLS]
                    )

                    roc_b, pr_b = vectorized_weighted_metrics(
                        y_true,
                        aug["y_prob"].to_numpy(),
                        W,
                    )

                    seed_roc_boot.append(roc_b)
                    seed_pr_boot.append(pr_b)

                aug_mean_roc_boot = np.vstack(
                    seed_roc_boot
                ).mean(axis=0)

                aug_mean_pr_boot = np.vstack(
                    seed_pr_boot
                ).mean(axis=0)

                delta_roc_boot = (
                    aug_mean_roc_boot - base_roc_boot
                )

                delta_pr_boot = (
                    aug_mean_pr_boot - base_pr_boot
                )

                point = paired_effects.loc[
                    (paired_effects["landmark"] == landmark)
                    & (paired_effects["model"] == model_name)
                    & (paired_effects["method"] == method)
                    & (paired_effects["ratio"] == ratio)
                ]

                assert len(point) == 1
                point = point.iloc[0]

                bootstrap_rows.append(
                    {
                        "landmark": landmark,
                        "model": model_name,
                        "method": method,
                        "ratio": ratio,
                        "delta_roc": float(
                            point["delta_roc"]
                        ),
                        "roc_ci_low": float(
                            np.percentile(
                                delta_roc_boot,
                                2.5,
                            )
                        ),
                        "roc_ci_high": float(
                            np.percentile(
                                delta_roc_boot,
                                97.5,
                            )
                        ),
                        "delta_pr": float(
                            point["delta_pr"]
                        ),
                        "pr_ci_low": float(
                            np.percentile(
                                delta_pr_boot,
                                2.5,
                            )
                        ),
                        "pr_ci_high": float(
                            np.percentile(
                                delta_pr_boot,
                                97.5,
                            )
                        ),
                        "n_boot_requested": N_BOOT,
                        "valid_boot": int(
                            len(valid_boot)
                        ),
                        "bootstrap_seed": BOOT_SEED,
                    }
                )

boot_results = pd.DataFrame(bootstrap_rows)

assert len(boot_results) == 72
assert (
    boot_results
    .groupby("landmark")["valid_boot"]
    .nunique()
    .eq(1)
    .all()
)

display(
    boot_results.loc[
        (
            (boot_results["model"] == "random_forest")
            & (boot_results["landmark"].isin([30, 40, 60]))
        )
        |
        (
            (boot_results["model"] == "logistic")
            & (boot_results["landmark"].isin([30, 40]))
        )
    ]
    .sort_values(
        ["model", "landmark", "method", "ratio"]
    )
    .round(6)
)


## 14. No-cherry-picking audit

All prespecified methods, ratios, and seeds are retained regardless of effect direction. The inferential table contains all 72 model × landmark × augmentation-method × ratio contrasts.


In [ ]:
observed_methods = set(
    full_oof.loc[
        full_oof["method"] != "none",
        "method",
    ].unique()
)

observed_ratios = set(
    full_oof.loc[
        full_oof["method"] != "none",
        "ratio",
    ].astype(float).unique()
)

observed_seeds = set(
    full_oof.loc[
        full_oof["method"] != "none",
        "seed",
    ].astype(int).unique()
)

assert observed_methods == set(AUGMENTATION_METHODS)
assert observed_ratios == set(AUGMENTATION_RATIOS)
assert observed_seeds == set(AUGMENTATION_SEEDS)
assert len(boot_results) == 72

print("All prespecified augmentation arms retained.")
print("Null and negative effects retained.")
print("No post-hoc best-arm selection used for primary inference.")


## 15. Leave-one-positive-athlete-out descriptive sensitivity

No models are retrained. For each frozen OOF contrast, one positive athlete is removed from evaluation at a time to assess whether the observed direction is driven by a single positive athlete.


In [ ]:
positive_athletes = (
    full_oof.loc[
        (full_oof["method"] == "none")
        & (full_oof["y_true"] == 1),
        "player_name",
    ]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

assert len(positive_athletes) == 5

lopo_rows = []

for landmark in LANDMARKS:
    for model_name in AUGMENTATION_MODELS:
        base = (
            full_oof.loc[
                (full_oof["landmark"] == landmark)
                & (full_oof["method"] == "none")
                & (full_oof["model"] == model_name)
            ]
            .sort_values(KEY_COLS)
            .reset_index(drop=True)
        )

        y_true = base["y_true"].to_numpy()
        base_prob = base["y_prob"].to_numpy()
        players = base["player_name"].to_numpy()

        for method in AUGMENTATION_METHODS:
            for ratio in AUGMENTATION_RATIOS:
                aug_by_seed = {}

                for seed in AUGMENTATION_SEEDS:
                    aug = (
                        full_oof.loc[
                            (full_oof["landmark"] == landmark)
                            & (full_oof["method"] == method)
                            & (full_oof["ratio"] == ratio)
                            & (full_oof["seed"] == seed)
                            & (full_oof["model"] == model_name)
                        ]
                        .sort_values(KEY_COLS)
                        .reset_index(drop=True)
                    )

                    assert base[KEY_COLS].equals(
                        aug[KEY_COLS]
                    )

                    aug_by_seed[seed] = (
                        aug["y_prob"].to_numpy()
                    )

                for excluded_athlete in positive_athletes:
                    keep = players != excluded_athlete
                    y_sub = y_true[keep]

                    assert np.unique(y_sub).size == 2

                    base_roc = roc_auc_score(
                        y_sub,
                        base_prob[keep],
                    )
                    base_pr = average_precision_score(
                        y_sub,
                        base_prob[keep],
                    )

                    seed_rocs = [
                        roc_auc_score(
                            y_sub,
                            aug_by_seed[seed][keep],
                        )
                        for seed in AUGMENTATION_SEEDS
                    ]

                    seed_prs = [
                        average_precision_score(
                            y_sub,
                            aug_by_seed[seed][keep],
                        )
                        for seed in AUGMENTATION_SEEDS
                    ]

                    lopo_rows.append(
                        {
                            "landmark": landmark,
                            "model": model_name,
                            "method": method,
                            "ratio": ratio,
                            "excluded_positive_athlete": excluded_athlete,
                            "n_sessions": int(keep.sum()),
                            "n_positive_sessions": int(
                                y_sub.sum()
                            ),
                            "base_roc": float(base_roc),
                            "aug_roc_mean": float(
                                np.mean(seed_rocs)
                            ),
                            "delta_roc": float(
                                np.mean(seed_rocs)
                                - base_roc
                            ),
                            "base_pr": float(base_pr),
                            "aug_pr_mean": float(
                                np.mean(seed_prs)
                            ),
                            "delta_pr": float(
                                np.mean(seed_prs)
                                - base_pr
                            ),
                        }
                    )

lopo_results = pd.DataFrame(lopo_rows)

assert len(lopo_results) == 360

def sign_consistency(series):
    if (series > 0).all():
        return "all_positive"
    if (series < 0).all():
        return "all_negative"
    return "mixed"


lopo_summary = (
    lopo_results
    .groupby(
        ["landmark", "model", "method", "ratio"],
        as_index=False,
    )
    .agg(
        delta_roc_mean=("delta_roc", "mean"),
        delta_roc_min=("delta_roc", "min"),
        delta_roc_max=("delta_roc", "max"),
        delta_pr_mean=("delta_pr", "mean"),
        delta_pr_min=("delta_pr", "min"),
        delta_pr_max=("delta_pr", "max"),
    )
)

roc_sign = (
    lopo_results
    .groupby(
        ["landmark", "model", "method", "ratio"]
    )["delta_roc"]
    .apply(sign_consistency)
    .reset_index(name="roc_sign_consistency")
)

pr_sign = (
    lopo_results
    .groupby(
        ["landmark", "model", "method", "ratio"]
    )["delta_pr"]
    .apply(sign_consistency)
    .reset_index(name="pr_sign_consistency")
)

lopo_summary = (
    lopo_summary
    .merge(
        roc_sign,
        on=["landmark", "model", "method", "ratio"],
        validate="one_to_one",
    )
    .merge(
        pr_sign,
        on=["landmark", "model", "method", "ratio"],
        validate="one_to_one",
    )
)

display(
    lopo_summary.loc[
        (
            (lopo_summary["model"] == "random_forest")
            & (lopo_summary["landmark"].isin([30, 40, 60]))
        )
        |
        (
            (lopo_summary["model"] == "logistic")
            & (lopo_summary["landmark"].isin([30, 40]))
        )
    ]
    .sort_values(
        ["model", "landmark", "method", "ratio"]
    )
    .round(6)
)


## 16. Equal-athlete-weighted sensitivity

No models are retrained. Within each landmark, each athlete receives total evaluation weight 1, so athletes with more sessions do not contribute more total metric weight.


In [ ]:
equal_weight_rows = []

for landmark in LANDMARKS:
    for model_name in AUGMENTATION_MODELS:
        base = (
            full_oof.loc[
                (full_oof["landmark"] == landmark)
                & (full_oof["method"] == "none")
                & (full_oof["model"] == model_name)
            ]
            .sort_values(KEY_COLS)
            .reset_index(drop=True)
        )

        y_true = base["y_true"].to_numpy()
        base_prob = base["y_prob"].to_numpy()

        athlete_session_counts = (
            base["player_name"].value_counts()
        )

        equal_weights = (
            base["player_name"]
            .map(
                lambda athlete: (
                    1.0
                    / athlete_session_counts.loc[athlete]
                )
            )
            .to_numpy()
        )

        check_weights = (
            pd.DataFrame(
                {
                    "player_name": base["player_name"],
                    "weight": equal_weights,
                }
            )
            .groupby("player_name")["weight"]
            .sum()
        )

        assert np.allclose(
            check_weights.to_numpy(),
            1.0,
        )

        base_roc_equal = roc_auc_score(
            y_true,
            base_prob,
            sample_weight=equal_weights,
        )

        base_pr_equal = average_precision_score(
            y_true,
            base_prob,
            sample_weight=equal_weights,
        )

        for method in AUGMENTATION_METHODS:
            for ratio in AUGMENTATION_RATIOS:
                seed_rocs = []
                seed_prs = []

                for seed in AUGMENTATION_SEEDS:
                    aug = (
                        full_oof.loc[
                            (full_oof["landmark"] == landmark)
                            & (full_oof["method"] == method)
                            & (full_oof["ratio"] == ratio)
                            & (full_oof["seed"] == seed)
                            & (full_oof["model"] == model_name)
                        ]
                        .sort_values(KEY_COLS)
                        .reset_index(drop=True)
                    )

                    assert base[KEY_COLS].equals(
                        aug[KEY_COLS]
                    )

                    seed_rocs.append(
                        roc_auc_score(
                            y_true,
                            aug["y_prob"],
                            sample_weight=equal_weights,
                        )
                    )

                    seed_prs.append(
                        average_precision_score(
                            y_true,
                            aug["y_prob"],
                            sample_weight=equal_weights,
                        )
                    )

                equal_weight_rows.append(
                    {
                        "landmark": landmark,
                        "model": model_name,
                        "method": method,
                        "ratio": ratio,
                        "n_athletes": int(
                            base["player_name"].nunique()
                        ),
                        "n_sessions": len(base),
                        "base_roc_equal": float(
                            base_roc_equal
                        ),
                        "aug_roc_equal_mean": float(
                            np.mean(seed_rocs)
                        ),
                        "delta_roc_equal": float(
                            np.mean(seed_rocs)
                            - base_roc_equal
                        ),
                        "base_pr_equal": float(
                            base_pr_equal
                        ),
                        "aug_pr_equal_mean": float(
                            np.mean(seed_prs)
                        ),
                        "delta_pr_equal": float(
                            np.mean(seed_prs)
                            - base_pr_equal
                        ),
                    }
                )

equal_weight_results = pd.DataFrame(
    equal_weight_rows
)

assert len(equal_weight_results) == 72

comparison_15b = (
    paired_effects[
        [
            "landmark",
            "model",
            "method",
            "ratio",
            "delta_roc",
            "delta_pr",
        ]
    ]
    .merge(
        equal_weight_results[
            [
                "landmark",
                "model",
                "method",
                "ratio",
                "delta_roc_equal",
                "delta_pr_equal",
            ]
        ],
        on=["landmark", "model", "method", "ratio"],
        validate="one_to_one",
    )
)

comparison_15b["roc_same_direction"] = (
    np.sign(comparison_15b["delta_roc"])
    == np.sign(comparison_15b["delta_roc_equal"])
)

comparison_15b["pr_same_direction"] = (
    np.sign(comparison_15b["delta_pr"])
    == np.sign(comparison_15b["delta_pr_equal"])
)

print(
    "ROC direction agreement:",
    f"{comparison_15b['roc_same_direction'].mean():.1%}",
)

print(
    "PR direction agreement:",
    f"{comparison_15b['pr_same_direction'].mean():.1%}",
)


## 17. Canonical export and runtime provenance


In [ ]:
performance_summary.to_csv(
    OUTPUT_DIR / "augmentation_seed_performance.csv",
    index=False,
)

aug_seed_summary.to_csv(
    OUTPUT_DIR / "augmentation_seed_aggregated_summary.csv",
    index=False,
)

paired_effects.to_csv(
    OUTPUT_DIR / "augmentation_paired_mean_effects.csv",
    index=False,
)

boot_results.to_csv(
    OUTPUT_DIR / "augmentation_cluster_bootstrap.csv",
    index=False,
)

lopo_results.to_csv(
    OUTPUT_DIR / "bullet15a_lopo_augmentation_effects.csv",
    index=False,
)

lopo_summary.to_csv(
    OUTPUT_DIR / "bullet15a_lopo_augmentation_summary.csv",
    index=False,
)

equal_weight_results.to_csv(
    OUTPUT_DIR / "bullet15b_equal_athlete_augmentation.csv",
    index=False,
)

comparison_15b.to_csv(
    OUTPUT_DIR / "bullet15b_primary_vs_equal_athlete.csv",
    index=False,
)

ctgan_quality_summary.to_csv(
    OUTPUT_DIR / "ctgan_quality_audit.csv",
    index=False,
)

runtime_versions = pd.DataFrame(
    [
        {"package": "python", "version": platform.python_version()},
        {"package": "numpy", "version": np.__version__},
        {"package": "pandas", "version": pd.__version__},
        {"package": "scikit-learn", "version": sklearn.__version__},
        {
            "package": "imbalanced-learn",
            "version": imblearn.__version__,
        },
        {"package": "sdv", "version": sdv.__version__},
        {"package": "torch", "version": torch.__version__},
        {
            "package": "cuda_available",
            "version": str(torch.cuda.is_available()),
        },
        {
            "package": "gpu",
            "version": (
                torch.cuda.get_device_name(0)
                if torch.cuda.is_available()
                else "CPU"
            ),
        },
        {
            "package": "experiment_fingerprint",
            "version": EXPERIMENT_FINGERPRINT,
        },
        {
            "package": "primary_feature_fingerprint",
            "version": PRIMARY_FEATURE_FINGERPRINT,
        },
        {
            "package": "fold_assignment_fingerprint",
            "version": FOLD_ASSIGNMENT_FINGERPRINT,
        },
    ]
)

runtime_versions.to_csv(
    OUTPUT_DIR / "augmentation_runtime_versions.csv",
    index=False,
)

with open(
    OUTPUT_DIR / "augmentation_experiment_config.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        checkpoint_metadata,
        file,
        indent=2,
        sort_keys=True,
        default=str,
    )

print("Canonical augmentation artifacts saved to:", OUTPUT_DIR)


## Output contract and claims discipline

A successful run produces the full real held-out OOF checkpoint and canonical summary artifacts under `results/augmentation/`, including seed-level performance, seed-aggregated effects, shared-draw athlete-cluster bootstrap intervals, CTGAN quality diagnostics, leave-one-positive-athlete-out sensitivity, equal-athlete weighting, runtime versions, and experiment metadata.

The defensible interpretation is intentionally narrow:

- synthetic observations are training interventions, not new independent evidence;
- performance is evaluated only on real athlete-disjoint held-out sessions;
- CTGAN quality is imperfect and should not be described as faithful reconstruction of the positive distribution;
- augmentation utility is learner-, landmark-, and in some settings estimand-dependent;
- null and negative augmentation effects are retained;
- the 72 bootstrap contrasts are exploratory and are not multiplicity-adjusted confirmatory tests;
- exact within-session injury onset remains unknown;
- no augmentation result supports minute-specific prospective injury-risk or causal claims.
